# Build `nordic.zarr` for the web viewer

Loads MITgcm output, derives fields, regrids to a 320x312 lon/lat grid, and writes a **web-optimised** Zarr store for nordicseas3d.github.io.



In [ ]:
run /gpfs/home/bve23zsu/PhDing/github/PhDing/Utils/utils_29032026.ipynb

In [ ]:
%%time
# load model data from 1996 to 2016
ds=load_model_outputs(dir ="/gpfs/home/bve23zsu/nordic/nordic2.out",start_itr=5760000, end_itr=6105600)  # last restart timestamp 2016-09-29 itr 8121600

In [ ]:
rho = xr.apply_ufunc(
    densjmd95,
    ds.S.where(ds.maskC).astype("float32"),
    ds.T.where(ds.maskC).astype("float32"),
    np.float32(0),
    dask="parallelized",
    output_dtypes=[np.float32],
)

rho = (rho - np.float32(1000)).astype("float32")

rho.name = 'sigma0'
rho.attrs['units'] = 'kg m^-3'
rho.attrs['standard_name'] = 'Potential density anomaly'
rho.attrs['long_name'] = 'Potential density anomaly referenced to surface (σ₀)'
ds['rho'] =rho

In [ ]:
metrics = {
    ('X'): ['dxC', 'dxG', 'dxF', 'dxV'], # X distances
    ('Y'): ['dyC', 'dyG', 'dyF', 'dyU'], # Y distances
    ('Z'): ['drF', 'drW', 'drS', 'drC'], # Z distances
    ('X', 'Y'): ['rAw', 'rAs', 'rA', 'rAz'] # Areas in x-y plane
}

grid = Grid(ds, metrics=metrics,periodic=('X', 'Y'))
Eta_noice=((ds['sIceLoad'].where(ds.maskInC)/1027)+ds['Eta'])
ds['Eta_noice'] =Eta_noice


In [ ]:
ds['uwind_stress'], ds['vwind_stress']= grid.interp(ds.oceTAUX,'X').where(ds.maskInC).compute(), grid.interp(ds.oceTAUY,'Y').where(ds.maskInC).compute()
ds['U_cgrid'], ds['V_cgrid']= grid.interp(ds.U,'X').where(ds.maskC).compute(), grid.interp(ds.V,'Y').where(ds.maskC).compute()



In [ ]:
import xesmf as xe
from PIL import Image



In [ ]:
# Select region and time first
nordic = ds.sel(time='2010')#.sel(XC=slice(-30, 23), YC=slice(68.5, 81.5), )
vars_keep = ["S", "T", "rho", "SIarea", "uwind_stress", "vwind_stress"]  # U_cgrid/V_cgrid not used by the web viewer
nordic = nordic[vars_keep]
# Apply tracer mask to T and S
nordic["T"] = nordic["T"].where(ds.maskC)
nordic["S"] = nordic["S"].where(ds.maskC)
nordic["rho"] = nordic["rho"].where(ds.maskC)

In [ ]:
import xesmf as xe
from PIL import Image

lon1, lon2 = -30, 23
lat1, lat2 = 57.670002, 81.49752
nx, ny = 320, 312   # safe, fast, high quality

lon = np.linspace(lon1, lon2, nx)
lat = np.linspace(lat1, lat2, ny)
lon2d, lat2d = np.meshgrid(lon, lat)

# Target lon/lat grid
dst = xr.Dataset(
    { "lon": (("lat", "lon"), lon2d),
        "lat": (("lat", "lon"), lat2d),  })
regridder = xe.Regridder(nordic.where(ds.maskInC), dst, "patch", )
nordic_regrid = regridder(nordic)

In [ ]:
size_mb = nordic_regrid.nbytes / (1024 ** 2)
print(f"Dataset size: {size_mb:.2f} MB")

## Web-optimised export

- **float32** everywhere (xesmf returns float64) -- ~2.2x smaller
- **bit-round** T/S/rho -- ~2x more, visually lossless, no viewer change
- **drop U_cgrid/V_cgrid** -- the viewer never reads them (done in `vars_keep`)


In [ ]:
# =============================================================================
# Web-optimised export of nordic.zarr  (replaces the old save cell)
#   * float32 everywhere (xesmf returns float64) .............. ~2.2x smaller
#   * bit-round T/S/rho (visually lossless, reader-transparent)  ~2x more
#   * U_cgrid/V_cgrid already dropped in vars_keep ............. (free, lossless)
#   * Blosc/zstd clevel=5; chunk per (time, depth) => 1 map slice = 1 chunk
# Set KEEPBITS = {} for pure lossless float32 (no bit-rounding).
# =============================================================================
import numpy as np
import numcodecs
import xarray as xr

dst        = "nordic.zarr"
SCALARS_4D = ["T", "S", "rho"]
FIELDS_3D  = ["SIarea", "uwind_stress", "vwind_stress"]
COORDS     = ["lon", "lat", "Z", "time"]
KEEPBITS   = {"T": 12, "S": 14, "rho": 14}   # ~0.004 degC / ~0.002 PSU / ~0.002 kg m^-3


def bitround(a, keepbits):
    """Zero the low mantissa bits of float32 (IEEE round-to-nearest).
    Dtype stays float32 so the browser reader needs no change; NaNs preserved.
    Absolute error <= ~ value / 2**keepbits."""
    x = np.ascontiguousarray(a, dtype=np.float32)
    finite = np.isfinite(x)
    xi = x.view(np.int32)
    keep = np.int32(~((1 << (23 - keepbits)) - 1))
    half = np.int32(1 << (23 - keepbits - 1))
    rounded = ((xi + half) & keep).view(np.float32)
    out = x.copy()
    out[finite] = rounded[finite]
    return out


# keep only what the viewer reads, plus coordinates (proven selection pattern)
keep = SCALARS_4D + FIELDS_3D + COORDS
web = nordic_regrid[[v for v in keep if v in nordic_regrid.variables]]

# float32 everywhere + clear inherited encoding (avoids chunk/dtype conflicts)
for v in list(web.data_vars):
    if np.issubdtype(web[v].dtype, np.floating):
        web[v] = web[v].astype("float32")
    web[v].encoding = {}

# bit-round the scalars in place (dask-safe; browser reads plain float32)
for v in SCALARS_4D:
    if v in web and KEEPBITS.get(v):
        web[v] = xr.apply_ufunc(
            bitround, web[v], kwargs={"keepbits": KEEPBITS[v]},
            dask="parallelized", output_dtypes=[np.float32], keep_attrs=True,
        )

web = web.chunk({"time": 1, "Z": 1, "lat": -1, "lon": -1})
ny, nx = web.sizes["lat"], web.sizes["lon"]
comp = numcodecs.Blosc(cname="zstd", clevel=5, shuffle=numcodecs.Blosc.SHUFFLE)
enc = {v: {"chunks": (1, 1, ny, nx), "compressor": comp, "dtype": "float32"}
       for v in SCALARS_4D if v in web}
enc.update({v: {"chunks": (1, ny, nx), "compressor": comp, "dtype": "float32"}
            for v in FIELDS_3D if v in web})

web.to_zarr(dst, mode="w", consolidated=True, encoding=enc, zarr_version=2)
print("wrote", dst, "->", list(web.data_vars))

In [ ]:
# --- verify the new store: on-disk size, file count, dtypes -------------------
import os
import xarray as xr

def _dirsize(p):
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(p) for f in fs)

nfiles = sum(len(fs) for _, _, fs in os.walk("nordic.zarr"))
print(f"on-disk nordic.zarr: {_dirsize('nordic.zarr')/1e9:.2f} GB  ({nfiles} files)")

chk = xr.open_zarr("nordic.zarr", consolidated=True)
for v in list(chk.data_vars):
    print(f"  {v:14s} {str(chk[v].dtype):8s} {tuple(chk[v].shape)}")
# sanity: values should be finite over ocean
print("T finite range:", float(chk['T'].min()), "to", float(chk['T'].max()))